In [2]:
from langchain.tools import tool
@ tool
def search_duckduck_go(query:str):
    """
    This tool will help you find latest News , Information , Updates 
    """
    from langchain_community.tools import DuckDuckGoSearchRun
    from langchain_community.utilities import DuckDuckGoSearchAPIWrapper
    
    duck_search=DuckDuckGoSearchRun(api_wrapper=DuckDuckGoSearchAPIWrapper())
    
    
    return duck_search.invoke(query)

@tool
def wiki_tool(query:str):
    """
    This tool will help you find information about a person , Object 
    """
    from langchain_community.tools import WikipediaQueryRun
    from langchain_community.utilities import WikipediaAPIWrapper

    wiki_querry=WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

    return wiki_querry.invoke(query)

@tool
def arxiv_tool(query:str):
    """This tool will be help to find the research papers or information from
       Research Papers..    
    """
    from langchain_community.tools import ArxivQueryRun
    from langchain_community.utilities import ArxivAPIWrapper

    arxiv_querry=ArxivQueryRun(api_wrapper=ArxivAPIWrapper())

    return arxiv_querry.invoke(query)




# **Custom Tool**

In [3]:
@tool
def personal_info(name:str):
    "This tool will help you find infomation of Fairoz and Arshid"
    info={
        "Fairoz":"This is Me. ",
        "Arshid":"He is a Good Freind.",
    }
    return info.get(name,"I Am not aware of this guy yet")

# **Tool Binding**
Not every model allows tool binding search documentation for checking if your model allows 

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import find_dotenv,load_dotenv
import os 
if os.environ["GOOGLE_API_KEY"]:
    print("API key is present ")
else:
    print("Please Provide an api key ")

API key is present 


In [5]:
llm =ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')

In [6]:
tools=[arxiv_tool,wiki_tool,search_duckduck_go,personal_info]

llm_with_tools=llm.bind_tools(tools)

In [7]:
response=llm_with_tools.invoke("Who is Arshid ?")


>The below content shows that as i am asking info of my freind now LLM wont generate its own answer that is the reason LLM's Response is empty and it has something in response.toolcalls

In [8]:
print(response)

content='' additional_kwargs={'function_call': {'name': 'personal_info', 'arguments': '{"name": "Arshid"}'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019de49d-0a69-74e0-9171-7b5387a9a79f-0' tool_calls=[{'name': 'personal_info', 'args': {'name': 'Arshid'}, 'id': '73616454-46b7-4525-be0d-24f4e83b4183', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 195, 'output_tokens': 16, 'total_tokens': 211, 'input_token_details': {'cache_read': 0}}


In [9]:
print(response.tool_calls)

[{'name': 'personal_info', 'args': {'name': 'Arshid'}, 'id': '73616454-46b7-4525-be0d-24f4e83b4183', 'type': 'tool_call'}]


# **React Agents**

In [10]:
# Create Schema
from typing import TypedDict,List

class graph_schema(TypedDict):
      messages:List
    

In [12]:
from langchain_core.prompts import ChatPromptTemplate

# **Node functions**

In [ ]:


# LLM NODE

def llm_node(state:graph_schema)-> graph_schema:
    messages=state['messages']
    prompt=ChatPromptTemplate.from_messages([
        ('system',"You are a helpfull asssistant that use tools to answer few questions "),
        ('human',{input})
    ])
    chain=prompt | llm_with_tools

    response=chain.invoke({'human':messages})

    state['messages']=messages+[response]

    return state


In [15]:
from langchain_core.messages import ToolMessage

In [16]:
# Tool Node
def tool_node(state:graph_schema)->graph_schema:
    
    messages=state['messages']
    
    tools_by_name={tool.name : tool for tool in tools}
    tool_results=[]

    for tool_call in messages[-1].tool_calls:
        tool=tools_by_name[tool_call['name']]
        observation=tool.invoke(tool_call['args'])
        tool_results.append(ToolMessage(content=observation,tool_call_id=tool_call['id']))
    
    state['messages']=messages + tool_results

    return state


# **Define coditional edge** 

In [ ]:
def is_tool_call(state:graph_schema)->str:
    last_message=state['messages'][-1]
    if last_message.tool_calls:
        return "tool_node"
    else:
        return "end"